# Keenable quickstart

[Keenable](https://keenable.ai) is a web search API built for AI agents. Search returns ranked pages with their text already extracted, and fetch returns any URL as clean markdown.

This notebook needs **no account and no API key**: every call below works keyless. Run the cells from top to bottom (in Colab: Runtime > Run all).

SDK source: [keenableai/keenable-sdk](https://github.com/keenableai/keenable-sdk) · Docs: [docs.keenable.ai](https://docs.keenable.ai)

In [ ]:
%pip install --quiet keenable

## Search

`Keenable()` with no arguments is a keyless client. Each result carries the page text in `snippet`.

In [ ]:
from keenable import Keenable

keenable = Keenable()

results = keenable.search("how wafer scale chips avoid the memory bottleneck", max_results=5)

for result in results:
    print(result.title)
    print(result.url)
    print((result.snippet or "")[:200], "\n")

Filters are optional and combine freely: `site`, `exclude_domains`, `published_after` / `published_before`, `max_results`, `snippet_max_length` and more.

In [ ]:
recent = keenable.search(
    "post-training quantization results",
    site="arxiv.org",
    published_after="2026-01-01",
    max_results=3,
)
for result in recent:
    print(result.published_at, result.title, result.url)

## Ground a model in one line

`to_context()` renders the results as a numbered, citable block you can drop straight into a prompt.

In [ ]:
context = results.to_context(max_chars=3000)
print(context)

prompt = f"Answer using only these sources, citing them by number.\n\n{context}\n\nQuestion: ..."

## Read a full page

`fetch()` returns each URL as clean markdown. A URL that failed has `error` set instead of raising.

In [ ]:
response = keenable.fetch(["https://en.wikipedia.org/wiki/Web_search_engine"], max_chars=2000)

for page in response.results:
    if page.error:
        print(page.url, "failed:", page.error.code)
        continue
    print(page.title)
    print(page.content[:1000])

## Tool definitions for an LLM

`TOOLS` holds OpenAI-compatible tool definitions (`keenable_search` and `keenable_fetch`) for any inference API that speaks that schema. When the model calls one, `run_tool_call` runs it and returns text ready for the `tool` message. Here we run a call by hand, with the arguments a model would send.

In [ ]:
import json

from keenable import TOOLS, run_tool_call

for tool in TOOLS:
    print(tool["function"]["name"], "-", tool["function"]["description"][:80], "...")

arguments = json.dumps({"query": "latest open-weight language model releases"})
print(run_tool_call(keenable, "keenable_search", arguments)[:1500])

## Handling errors

The SDK already retries what can be retried, so an error that reaches you is final for that call. Every `KeenableAPIError` carries what the API says about the failure: branch on `code`, read `fix` for what to do next, and quote `request_id` when reporting a problem.

Here the API rejects a path in `exclude_domains` with HTTP 400, on purpose.

In [ ]:
from keenable import KeenableAPIError, KeenableRateLimitError

try:
    keenable.search("monstera repotting", exclude_domains=["example.com/blog"])
except KeenableRateLimitError as err:
    # Keyless calls share an hourly cap per IP address; an API key lifts it.
    print(f"rate limited, retry in {err.retry_after_seconds}s:", err.fix)
except KeenableAPIError as err:
    print("status_code:", err.status_code)
    print("code:       ", err.code)
    print("type:       ", err.type)
    print("fix:        ", err.fix)
    print("retryable:  ", err.retryable)
    print("param:      ", err.param)
    print("value:      ", err.value)
    print("reason:     ", err.reason)
    print("request_id: ", err.request_id)

## Add an API key (optional)

Keyless calls have an hourly cap per IP address. An API key lifts it and comes with 100,000 free requests a month: create one in the [Keenable console](https://app.keenable.ai/console). In Colab, save it as a secret named `KEENABLE_API_KEY` (the key icon in the left sidebar) and run the cell below. Elsewhere, set the `KEENABLE_API_KEY` environment variable, or pass the key directly: `Keenable("keen_...")`.

In [ ]:
import os

try:
    from google.colab import userdata

    os.environ["KEENABLE_API_KEY"] = userdata.get("KEENABLE_API_KEY")
except Exception:
    pass  # not in Colab, or no secret saved: the client stays keyless

keenable = Keenable()  # picks up KEENABLE_API_KEY when it is set
print("using an API key" if os.environ.get("KEENABLE_API_KEY") else "keyless")

## Next steps

- [Python SDK reference](https://github.com/keenableai/keenable-sdk/tree/main/python): async client, monitors, retries and timeouts
- [All four SDKs](https://github.com/keenableai/keenable-sdk): Python, TypeScript, Go and Rust
- [Examples](https://github.com/keenableai/keenable-sdk/tree/main/examples): tool calling, grounded answers, domain filters, monitors
- [Documentation](https://docs.keenable.ai)